# Lab 03.1 — HDFS e Spark

## Introduzione

**HDFS** (Hadoop Distributed File System) memorizza file molto grandi su più macchine:

* ogni file è diviso in **blocchi** (qui 16 MB, in produzione 128 MB);
* ogni blocco è **replicato** su più **DataNode** (qui 2 copie) → tolleranza ai guasti;
* il **NameNode** conserva solo i metadati: albero delle directory e posizione dei blocchi.

Spark legge HDFS in parallelo: **un blocco → una partizione → un task**, eseguito possibilmente sul nodo che ha il blocco (*data locality*).

Web UI del NameNode: http://localhost:9870 (menu *Utilities → Browse the file system*).

### Obiettivi

1. Avviare la `SparkSession` e verificare che il filesystem di default sia HDFS.
2. Caricare un file su HDFS con la API REST **WebHDFS**.
3. Usare la API **FileSystem** di Hadoop dalla JVM di Spark.
4. Osservare **blocchi e repliche** di un file scritto da Spark.
5. Capire il legame **blocco HDFS → partizione Spark**.
6. Eseguire un **Word Count** con input e output su HDFS.
7. Scrivere **Parquet partizionato** su HDFS e interrogarlo con Spark SQL.
8. Distinguere **file locali e HDFS** (schema `file://` vs percorso senza schema).

## Step 1: Creazione della SparkSession

La funzione `crea_sessione` (modulo `sessione.py`, nella stessa cartella) legge la variabile d'ambiente `SPARK_MASTER` e crea la sessione con il master giusto: Standalone (default) oppure YARN. Il driver trova HDFS grazie a `HADOOP_CONF_DIR`, quindi stampiamo master e `fs.defaultFS`: il secondo deve essere `hdfs://namenode:8020`, cioè i percorsi **senza schema** punteranno a HDFS.

In [ ]:
import os, requests
from pyspark.sql import functions as F
from sessione import crea_sessione  # helper del laboratorio: sceglie il master in base alla modalità di avvio

spark = crea_sessione("Lab03-HDFS")
sc = spark.sparkContext
print("Master:", sc.master)
# fs.defaultFS è il filesystem usato quando un percorso non ha schema (hdfs://, file://, ...)
print("Filesystem di default:", sc._jsc.hadoopConfiguration().get("fs.defaultFS"))

## Step 2: Caricare un file su HDFS con WebHDFS (API REST)

WebHDFS espone HDFS via HTTP, senza librerie Hadoop. La scrittura avviene in **due passi**: il NameNode, che ha solo i metadati, risponde con un **redirect** verso il DataNode che riceverà i dati; il client invia poi il contenuto direttamente a quel DataNode. Alla fine `GETFILESTATUS` mostra lunghezza, fattore di replica e dimensione del blocco.

In [ ]:
WEBHDFS = "http://namenode:9870/webhdfs/v1"
UTENTE = {"user.name": "jovyan"}

# Crea la directory /dati su HDFS (operazione sui metadati: risponde il NameNode)
requests.put(f"{WEBHDFS}/dati", params={**UTENTE, "op": "MKDIRS"}).raise_for_status()

locale = "/home/jovyan/datasets/alice_in_wonderland.txt"
# Passo 1: il NameNode non riceve i dati, risponde con un redirect verso un DataNode
r1 = requests.put(f"{WEBHDFS}/dati/alice.txt", params={**UTENTE, "op": "CREATE", "overwrite": "true"},
                  allow_redirects=False)
print("1) NameNode risponde", r1.status_code, "→ redirect a", r1.headers["Location"].split("?")[0])
# Passo 2: i dati vanno inviati al DataNode indicato nell'header Location
with open(locale, "rb") as f:
    r2 = requests.put(r1.headers["Location"], data=f)
print("2) DataNode risponde", r2.status_code, "(Created)")

# Metadati del file appena creato: lunghezza, replica, dimensione del blocco, proprietario
stato = requests.get(f"{WEBHDFS}/dati/alice.txt", params={**UTENTE, "op": "GETFILESTATUS"}).json()["FileStatus"]
print({k: stato[k] for k in ("length", "replication", "blockSize", "owner")})

## Step 3: La API FileSystem di Hadoop (dalla JVM di Spark)

Spark accede ai file attraverso la API `FileSystem` di Hadoop, che qui richiamiamo direttamente tramite il gateway Py4J (`sc._jvm`). È la stessa API usata internamente da Spark e funziona con qualunque filesystem compatibile Hadoop (`hdfs://`, `ofs://`, `s3a://`, `file://`). Definiamo una piccola funzione `ls` che useremo anche nei prossimi step.

In [ ]:
jvm = sc._jvm
conf = sc._jsc.hadoopConfiguration()                 # configurazione Hadoop del driver
Path = jvm.org.apache.hadoop.fs.Path
fs = jvm.org.apache.hadoop.fs.FileSystem.get(conf)   # FileSystem di default (HDFS)

def ls(percorso):
    # stampa: tipo (d/-), fattore di replica, dimensione in byte, percorso
    for st in fs.listStatus(Path(percorso)):
        tipo = "d" if st.isDirectory() else "-"
        print(f"{tipo} {st.getReplication()} {st.getLen():>12,d}  {st.getPath().toUri().getPath()}")

ls("/dati")

### Punti chiave

* Il **NameNode** gestisce solo i metadati: i dati passano dai **DataNode** (da qui il redirect di WebHDFS).
* Un file HDFS ha una **replica** (2 in questo lab) e una **dimensione di blocco** (16 MB) configurabili.
* Spark usa la API `FileSystem` di Hadoop: lo stesso codice vale per HDFS, Ozone, S3 e file locali.

## Step 4: Blocchi e repliche

Scriviamo con Spark un file più grande (~ 70 MB): `coalesce(1)` forza un unico file di output, così HDFS lo divide in più blocchi da 16 MB. Con `getFileBlockLocations` leggiamo, per ogni blocco, l'offset, la lunghezza e i **DataNode** che ne conservano una copia (2 per blocco). Il percorso `/dati/grande` non ha schema, quindi finisce su HDFS.

In [ ]:
# 1 milione di righe con un hash SHA-256 per renderle poco comprimibili (~ 70 MB di CSV)
grande = (spark.range(0, 1_000_000)
          .select(F.col("id"),
                  F.sha2(F.col("id").cast("string"), 256).alias("hash"))
          .coalesce(1))                                  # un solo file, per vedere i blocchi
grande.write.mode("overwrite").option("header", True).csv("/dati/grande")   # path senza schema → HDFS

# Individua il file part-* prodotto da Spark e leggine i metadati
file_csv = [st.getPath() for st in fs.listStatus(Path("/dati/grande")) if st.getPath().getName().startswith("part-")][0]
st = fs.getFileStatus(file_csv)
print(f"{file_csv.getName()}: {st.getLen() / 1e6:.1f} MB, blocco {st.getBlockSize() // 2**20} MB, replica {st.getReplication()}\n")
# Per ogni blocco: posizione nel file e DataNode che ne hanno una copia
for i, b in enumerate(fs.getFileBlockLocations(st, 0, st.getLen())):
    print(f"blocco {i}: offset {b.getOffset() / 1e6:6.1f} MB  lunghezza {b.getLength() / 1e6:5.1f} MB  → DataNode {list(b.getHosts())}")

## Step 5: Un blocco = una partizione

Leggendo il file con la API RDD, Spark crea una partizione (e quindi un task) per ogni blocco HDFS: il parallelismo cresce con la dimensione del file. La API DataFrame invece raggruppa i blocchi in partizioni da `spark.sql.files.maxPartitionBytes` (default 128 MB). Confrontiamo i tre casi, riducendo il parametro a 16 MB per ritrovare una partizione per blocco.

In [ ]:
# API RDD: una partizione per blocco HDFS (percorso con schema esplicito)
righe = sc.textFile("hdfs://namenode:8020/dati/grande")
print("RDD       → partizioni:", righe.getNumPartitions(), "(= blocchi HDFS)")

# API DataFrame: i blocchi vengono accorpati fino a maxPartitionBytes (128 MB di default)
df = spark.read.option("header", True).csv("/dati/grande")
print("DataFrame → partizioni:", df.rdd.getNumPartitions(), "(maxPartitionBytes = 128 MB)")

# Con 16 MB si torna a una partizione per blocco
spark.conf.set("spark.sql.files.maxPartitionBytes", "16m")
df = spark.read.option("header", True).csv("/dati/grande")
print("DataFrame → partizioni:", df.rdd.getNumPartitions(), "(maxPartitionBytes = 16 MB)")
print("Righe:", df.count())

### Punti chiave

* HDFS divide i file in blocchi e ne conserva **più copie su DataNode diversi**.
* In Spark **blocco → partizione → task**: più il file è grande, più il parallelismo di lettura cresce.
* Con i DataFrame il numero di partizioni dipende da `spark.sql.files.maxPartitionBytes`.

## Step 6: Word Count (MapReduce) con input e output su HDFS

Rieseguiamo il classico Word Count, questa volta leggendo `alice.txt` da HDFS (schema `hdfs://` esplicito) e scrivendo il risultato in `/dati/wordcount`. Prima di salvare cancelliamo la directory di output con la API `FileSystem` (Spark non sovrascrive directory esistenti); poi `ls` mostra i file `part-*` prodotti, uno per partizione, più `_SUCCESS`.

In [ ]:
import re
testo = sc.textFile("hdfs://namenode:8020/dati/alice.txt")
# map: ogni parola → (parola, 1); reduce: somma i conteggi per chiave
conteggi = (testo.flatMap(lambda l: re.findall(r"[a-z]+", l.lower()))
                 .map(lambda w: (w, 1))
                 .reduceByKey(lambda a, b: a + b))
print(conteggi.takeOrdered(10, key=lambda kv: -kv[1]))   # le 10 parole più frequenti

fs.delete(Path("/dati/wordcount"), True)   # saveAsTextFile fallisce se la directory esiste già
conteggi.map(lambda kv: f"{kv[0]}\t{kv[1]}").saveAsTextFile("/dati/wordcount")
ls("/dati/wordcount")

## Step 7: Parquet partizionato su HDFS e Spark SQL

Gli stessi concetti del Lab 02 valgono su un filesystem distribuito: cambia solo il percorso. Generiamo un milione di vendite, le scriviamo in Parquet partizionato per `regione` (una sotto-directory per regione) e le interroghiamo con SQL. Il filtro su `regione` permette a Spark di leggere solo le directory Sud e Isole (*partition pruning*).

In [ ]:
# Dataset sintetico: negozio (0-19), regione (una di 4) e importo casuale riproducibile (seed 7)
vendite = (spark.range(0, 1_000_000)
           .withColumn("negozio", (F.col("id") % 20).cast("int"))
           .withColumn("regione", F.element_at(F.array(*[F.lit(r) for r in ["Nord", "Centro", "Sud", "Isole"]]),
                                               (F.col("id") % 4 + 1).cast("int")))
           .withColumn("importo", F.round(F.rand(7) * 100, 2)))

# Una sotto-directory regione=<valore> per ogni regione
vendite.write.partitionBy("regione").mode("overwrite").parquet("/dati/vendite")
ls("/dati/vendite")

# Rilettura e query SQL: il filtro sulla colonna di partizione evita di leggere le altre regioni
spark.read.parquet("/dati/vendite").createOrReplaceTempView("vendite")
spark.sql("""
    SELECT regione, COUNT(*) AS scontrini, CAST(SUM(importo) AS DECIMAL(12, 2)) AS totale
    FROM vendite WHERE regione IN ('Sud', 'Isole')
    GROUP BY regione
""").show()

## Step 8: File locali vs HDFS

Con `fs.defaultFS=hdfs://namenode:8020` un percorso **senza schema** punta a HDFS. Per i file del container serve lo schema esplicito `file://` (e in un cluster il file deve esistere su **tutti** i nodi: per questo i dataset sono montati anche su worker e NodeManager). Contiamo le righe dello stesso testo nei due filesystem: il risultato deve coincidere.

In [ ]:
# file:/// → filesystem locale del container; senza schema → HDFS
print(spark.read.text("file:///home/jovyan/datasets/alice_in_wonderland.txt").count(), "righe dal filesystem locale")
print(spark.read.text("/dati/alice.txt").count(), "righe da HDFS")

### Punti chiave

* Un percorso senza schema usa `fs.defaultFS` (HDFS); per il disco locale serve `file://`.
* Il **Parquet partizionato** su HDFS si comporta come su disco locale, con in più blocchi replicati e distribuiti.
* L'output di Spark è sempre una **directory** di file `part-*`, uno per partizione.

## Step 9: Chiusura della sessione

`spark.stop()` termina l'applicazione e libera le risorse del cluster. I file creati su HDFS (`/dati/...`) restano: il notebook `02_spark_su_yarn` li riutilizza.

In [ ]:
spark.stop()  # termina l'applicazione Spark; i dati su HDFS restano disponibili

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| Filesystem di default | `hadoopConfiguration().get("fs.defaultFS")` | sapere dove puntano i percorsi senza schema |
| Scrittura via REST | WebHDFS `op=CREATE` + redirect al DataNode | caricare file su HDFS senza librerie Hadoop |
| Metadati del file | WebHDFS `GETFILESTATUS`, `fs.getFileStatus` | lunghezza, replica, dimensione del blocco |
| API FileSystem | `FileSystem.get(conf)`, `listStatus`, `delete` | gestire file su qualunque filesystem Hadoop |
| Blocchi e repliche | `getFileBlockLocations` | vedere su quali DataNode sono i blocchi |
| Blocco → partizione | `sc.textFile(...).getNumPartitions()` | parallelismo di lettura e data locality |
| Dimensione partizioni DataFrame | `spark.sql.files.maxPartitionBytes` | controllare le partizioni di lettura |
| Word Count su HDFS | `flatMap` / `reduceByKey` / `saveAsTextFile` | MapReduce con input e output distribuiti |
| Parquet partizionato | `write.partitionBy("regione").parquet(...)` | layout a directory e partition pruning |
| Locale vs HDFS | `file:///...` vs percorso senza schema | scegliere il filesystem corretto |